# Importação das Bibliotecas

In [1]:
import pandas as pd
import os
from datetime import datetime, date
from dateutil.relativedelta import relativedelta
from openpyxl import Workbook, load_workbook

# Carregando Base de Controle de Processos

In [2]:
id = 13
path_registros_processos = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'

registros_processos = pd.read_excel(path_registros_processos, sheet_name="REGISTROS", engine='openpyxl')
wb_p = load_workbook(path_registros_processos)
ws_p = wb_p['REGISTROS']

# Controle de atualização de processo: Etapa 0
linha_0 = [id, datetime.today(), 0]
ws_p.append(linha_0)
wb_p.save(path_registros_processos)

# Carregamento das Bases

In [3]:
colaboradores = pd.read_excel(r"X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\COLABORADORES.xlsx")
arquivo_unidades = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\DIMENSÕES POWER BI.xlsx'
unidades = pd.read_excel(arquivo_unidades, sheet_name='UNIDADES', usecols='B:F', skiprows=3)

arquivo_afastamentos = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\AFASTAMENTOS.xlsx'
afastamentos = pd.read_excel(arquivo_afastamentos)

arquivo_alteracoes_funcoes = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\ALTERAÇÕES DE FUNÇÕES_v2.xlsx'
alteracoes_funcoes = pd.read_excel(arquivo_alteracoes_funcoes)

arquivo_de_para_motivos_afastamentos = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\DE PARA - MOTIVO AFASTAMENTO.xlsx'
de_para_motivos_afastamentos = pd.read_excel(arquivo_de_para_motivos_afastamentos)

# Padronização preventiva de colunas (Evita o KeyError)
colaboradores.columns = ['Registro' if col.upper() == 'REGISTRO' else col for col in colaboradores.columns]
afastamentos.columns = ['Registro' if col.upper() == 'REGISTRO' else col for col in afastamentos.columns]
alteracoes_funcoes.columns = ['Registro' if col.upper() == 'REGISTRO' else col for col in alteracoes_funcoes.columns]

# Controle de atualização de processo: Etapa 1
linha_1 = [id, datetime.today(), 1]
ws_p.append(linha_1)
wb_p.save(path_registros_processos)

# Criação e Atribuição de Valores das Variáveis

In [4]:
hoje = datetime.today().date()
months = ["JAN", "FEV", "MAR", "ABR", "MAI", "JUN", "JUL", "AGO", "SET", "OUT", "NOV", "DEZ"]
mes_atual = months[datetime.today().month - 2]
ano_atual = datetime.today().year

mes = (datetime.today() - relativedelta(months=1)).month
ano = (datetime.today() - relativedelta(months=1)).year
primeiro_dia_mes = (datetime.today() - relativedelta(months=1)).replace(day=1).date()
ultimo_dia_mes = (datetime.today().replace(day=1) - relativedelta(days=1)).date()

manual = 0
if manual == 1:
    mes_atual = 'DEZ'
    mes = 12
    ano = 2026
    primeiro_dia_mes = date(2026, 12, 1)
    ultimo_dia_mes = date(2026, 12, 31)
    
# Controle de atualização de processo: Etapa 2
linha_2 = [id, datetime.today(), 2]
ws_p.append(linha_2)
wb_p.save(path_registros_processos)

# BASE 1 - Desligados no Mês de Referência

In [5]:
colaboradores['data_rescisao'] = pd.to_datetime(colaboradores['data_rescisao'])
relatorio_desligados = colaboradores.merge(unidades, left_on='cod_empresa', right_on='Código', how='inner')
relatorio_desligados = relatorio_desligados[
    (relatorio_desligados['data_rescisao'].dt.month == mes) & 
    (relatorio_desligados['data_rescisao'].dt.year == ano)
]
relatorio_desligados = relatorio_desligados[['Empresa', 'centro_custo', 'cargo_abreviado', 'data_rescisao', 'descricao_rescisao']]

# Controle de atualização de processo: Etapa 3
linha_3 = [id, datetime.today(), 3]
ws_p.append(linha_3)
wb_p.save(path_registros_processos)

# BASE 2 - Afastado no Mês de Referência

In [6]:
afastamentos_enviar = afastamentos.merge(de_para_motivos_afastamentos, left_on='descricao_afastamento', right_on='DE', how='inner')
afastamentos_enviar = afastamentos_enviar[afastamentos_enviar['ENVIAR'] == 'Sim'].copy()
afastamentos_enviar = afastamentos_enviar.rename(columns={'PARA': 'Descricao_resumo'})

afastamentos_enviar['data_afastamento'] = pd.to_datetime(afastamentos_enviar['data_afastamento'])
afastamentos_enviar['data_retorno'] = pd.to_datetime(afastamentos_enviar['data_retorno'])

data_inicio_calendar = afastamentos_enviar['data_afastamento'].min()
max_retorno = afastamentos_enviar['data_retorno'].max()
hoje_ts = pd.Timestamp(hoje)

if pd.isna(max_retorno) or max_retorno < hoje_ts:
    data_fim_calendar = hoje_ts
else:
    data_fim_calendar = max_retorno

calendario = pd.DataFrame({'Data': pd.date_range(start=data_inicio_calendar, end=data_fim_calendar, freq='D')})
registros_unicos = afastamentos_enviar[['Registro']].drop_duplicates()
registro_vs_datas = registros_unicos.merge(calendario, how='cross')

# Expansão dos dias de afastamento
linhas_afastados = []
for _, row in afastamentos_enviar.iterrows():
    if pd.notna(row['data_afastamento']):
        fim = row['data_retorno'] if pd.notna(row['data_retorno']) else pd.Timestamp('2030-12-31')
        if fim > data_fim_calendar:
            fim = data_fim_calendar
        if row['data_afastamento'] <= fim:
            datas = pd.date_range(start=row['data_afastamento'], end=fim)
            df_temp = pd.DataFrame({'Registro': row['Registro'], 'Data': datas})
            linhas_afastados.append(df_temp)

if linhas_afastados:
    afastados_dates = pd.concat(linhas_afastados, ignore_index=True).drop_duplicates()
    afastados_dates['Afastado'] = 1
else:
    afastados_dates = pd.DataFrame(columns=['Registro', 'Data', 'Afastado'])

afastados_dia = registro_vs_datas.merge(afastados_dates, on=['Registro', 'Data'], how='left')
afastados_dia['Afastado'] = afastados_dia['Afastado'].fillna(0).astype(int)

# Posição atual
posicao_atual = afastados_dia[afastados_dia['Data'] == hoje_ts]
posicao_atual_afastados = posicao_atual[posicao_atual['Afastado'] == 1][['Registro']]
posicao_atual_nao_afastados = posicao_atual[posicao_atual['Afastado'] == 0][['Registro']]

# Data afastamento dos atualmente afastados
temp1 = afastados_dia[afastados_dia['Registro'].isin(posicao_atual_afastados['Registro']) & (afastados_dia['Afastado'] == 0)]
ultimos_dias_nao_afastado = temp1.groupby('Registro')['Data'].max().reset_index()
ultimos_dias_nao_afastado['Data_afastamento'] = ultimos_dias_nao_afastado['Data'] + pd.Timedelta(days=1)
data_afastamento_afastado_data_atual = ultimos_dias_nao_afastado[['Registro', 'Data_afastamento']]

# Data afastamento dos não afastados atualmente
temp2 = afastados_dia[afastados_dia['Registro'].isin(posicao_atual_nao_afastados['Registro']) & (afastados_dia['Afastado'] == 1)]
atual_nao_afastado_ultimo_afastamento = temp2.groupby('Registro')['Data'].max().reset_index()

temp3 = afastados_dia[afastados_dia['Afastado'] == 0].merge(atual_nao_afastado_ultimo_afastamento, on='Registro', suffixes=('', '_ultimo_afast'))
temp3 = temp3[temp3['Data'] < temp3['Data_ultimo_afast']]
ultimos_dias_antes_afast = temp3.groupby('Registro')['Data'].max().reset_index()
ultimos_dias_antes_afast['Data_afastamento'] = ultimos_dias_antes_afast['Data'] + pd.Timedelta(days=1)
data_afastamento_nao_afastado_data_atual = ultimos_dias_antes_afast[['Registro', 'Data_afastamento']]

data_afastamento_consolidado = pd.concat([data_afastamento_afastado_data_atual, data_afastamento_nao_afastado_data_atual], ignore_index=True)

data_retorno_consolidado = afastamentos_enviar.sort_values('data_afastamento', ascending=False).drop_duplicates('Registro')
data_retorno_consolidado = data_retorno_consolidado[['Registro', 'data_retorno', 'Descricao_resumo']].rename(columns={'data_retorno': 'Data_retorno', 'Descricao_resumo': 'Motivo_afastamento'})

# Relatório consolidado
relatorio_afastados = data_retorno_consolidado.merge(colaboradores, on='Registro', how='inner')
relatorio_afastados = relatorio_afastados.merge(unidades, left_on='cod_empresa', right_on='Código', how='inner')
relatorio_afastados = relatorio_afastados.merge(data_afastamento_consolidado, on='Registro', how='inner')

primeiro_dia_mes_ts = pd.Timestamp(primeiro_dia_mes)
ultimo_dia_mes_ts = pd.Timestamp(ultimo_dia_mes)

mask_afast = (
    (relatorio_afastados['Data_afastamento'] <= ultimo_dia_mes_ts) &
    (relatorio_afastados['Data_retorno'].isna() | (relatorio_afastados['Data_retorno'] >= primeiro_dia_mes_ts)) &
    (relatorio_afastados['data_rescisao'].isna() | (pd.to_datetime(relatorio_afastados['data_rescisao']) >= primeiro_dia_mes_ts)) &
    (relatorio_afastados['Data_retorno'].isna() | ((relatorio_afastados['Data_retorno'] - relatorio_afastados['Data_afastamento']).dt.days > 15))
)
relatorio_afastados = relatorio_afastados[mask_afast]
relatorio_afastados = relatorio_afastados[['Empresa', 'centro_custo', 'Motivo_afastamento', 'Data_afastamento', 'Data_retorno']]
relatorio_afastados = relatorio_afastados.rename(columns={'centro_custo': 'Centro_de_custo'})

# Controle de atualização de processo: Etapa 4
linha_4 = [id, datetime.today(), 4]
ws_p.append(linha_4)
wb_p.save(path_registros_processos)

# BASE 3 - Alterações de Funções e Unidades

In [7]:
path_files = r'X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'
lista_arquivos = os.listdir(path_files)
df_arquivos = pd.DataFrame(lista_arquivos, columns=['arquivo'])
df_arquivos = df_arquivos[df_arquivos['arquivo'].str.upper().str.startswith('COLAB')]

def extrair_data_colab(nome_arquivo):
    try:
        nome_base = nome_arquivo.split('.')[0].upper()
        if '-' in nome_base and len(nome_base.split('-')[0]) > 12:
            mes, ano = int(nome_base[8:10]), int(nome_base[10:14])
        elif len(nome_base) >= 14:
            mes, ano = int(nome_base[8:10]), int(nome_base[10:14])
        else:
            return pd.NaT
        if not (1 <= mes <= 12): return pd.NaT
        return pd.to_datetime(f"{ano}-{mes:02d}-01", errors='coerce')
    except:
        return pd.NaT

df_arquivos['Data'] = df_arquivos['arquivo'].apply(extrair_data_colab)
df_arquivos = df_arquivos.dropna(subset=['Data']).sort_values(by='Data', ascending=True).reset_index(drop=True)
df_arquivos['Ordem'] = range(len(df_arquivos), 0, -1)

arquivo_mais_recente = df_arquivos[df_arquivos['Ordem'] == 1]
path_colaboradores_historico = os.path.join(path_files, arquivo_mais_recente['arquivo'].iloc[0])
colaboradores_historico = pd.read_excel(path_colaboradores_historico)
colaboradores_historico.columns = ['Registro' if col.upper() == 'REGISTRO' else col for col in colaboradores_historico.columns]

# 3.1 - Alterações de Unidades
colaboradores_transferidos_unicos = colaboradores_historico[colaboradores_historico['SITUACAO'] == 'T'][['Registro']].drop_duplicates()

colab_hist = colaboradores_historico.copy()
colab_hist['RESC_DAT_fill'] = pd.to_datetime(colab_hist['RESC_DAT']).fillna(pd.Timestamp('2050-12-31'))
colab_hist = colab_hist.sort_values(['Registro', 'RESC_DAT_fill'], ascending=[True, False])
colab_hist['Ordem'] = colab_hist.groupby('Registro').cumcount() + 1

transferencias_unidades = colab_hist.merge(unidades, left_on='COD_EMPRES', right_on='Código', how='inner')
transferencias_unidades = transferencias_unidades.merge(colaboradores_transferidos_unicos, on='Registro', how='inner')
transferencias_unidades = transferencias_unidades[['Registro', 'NOME_COMP', 'RESC_DAT', 'Empresa Resumo', 'CARGO_COMP', 'CCUSTO_CON', 'Ordem']]
transferencias_unidades.columns = ['Registro', 'nome', 'data_alteracao', 'unidade', 'cargo_completo', 'centro_custo', 'Ordem']

transferencias_B = transferencias_unidades.copy()
transferencias_B['Ordem_join'] = transferencias_B['Ordem'] - 1

alteracoes_unidades = transferencias_unidades.merge(transferencias_B, left_on=['Registro', 'Ordem'], right_on=['Registro', 'Ordem_join'], suffixes=('_destino', '_origem'))
alteracoes_unidades = alteracoes_unidades.rename(columns={
    'nome_destino': 'colaborador',
    'data_alteracao_origem': 'data_alteracao',
    'unidade_origem': 'unidade_origem',
    'unidade_destino': 'unidade_destino',
    'cargo_completo_origem': 'cargo_origem',
    'cargo_completo_destino': 'cargo_destino'
})
alteracoes_unidades = alteracoes_unidades[['Registro', 'colaborador', 'data_alteracao', 'unidade_origem', 'unidade_destino', 'cargo_origem', 'cargo_destino']]

# 3.2 - Alterações de Funções
alt_func = alteracoes_funcoes.copy()
alt_func['data_ref_atual'] = pd.to_datetime(alt_func['data_ref_atual'], errors='coerce', dayfirst=True)
alt_func = alt_func.sort_values('data_ref_atual')
alt_func['Rnk'] = alt_func.groupby(['cod_empresa', 'Registro', 'nome', 'cod_centro_custo', 'cargo_anterior', 'motivo_anterior', 'cargo', 'motivo_atual', 'data_ref_atual']).cumcount() + 1
alt_func_tratados = alt_func[alt_func['Rnk'] == 1].copy()

alt_func_tratados['N_ALT'] = alt_func_tratados.groupby('Registro').cumcount() + 1
alt_func_tratados = alt_func_tratados[['cod_empresa', 'Registro', 'nome', 'cod_centro_custo', 'cargo_anterior', 'motivo_anterior', 'cargo', 'data_ref_atual', 'N_ALT']]

# Converte a coluna 'Registro' para string em ambas as bases para garantir o merge
alt_func_tratados['Registro'] = alt_func_tratados['Registro'].astype(str)
colaboradores['Registro'] = colaboradores['Registro'].astype(str)

# Agora o merge
alt_func_mes = alt_func_tratados.merge(colaboradores[['Registro', 'data_admissao']], on='Registro', how='inner')
alt_func_mes['data_admissao'] = pd.to_datetime(alt_func_mes['data_admissao'])
alt_func_mes = alt_func_mes[
    (alt_func_mes['data_ref_atual'].dt.year == ano) &
    (alt_func_mes['data_ref_atual'].dt.month == mes) &
    (alt_func_mes['data_ref_atual'] != alt_func_mes['data_admissao'])
]

alt_func_tratados_B = alt_func_tratados.copy()
alt_func_tratados_B['N_ALT_join'] = alt_func_tratados_B['N_ALT'] + 1

relatorio_alteracoes = alt_func_mes.merge(alt_func_tratados_B, left_on=['Registro', 'N_ALT'], right_on=['Registro', 'N_ALT_join'], suffixes=('', '_anterior'))
relatorio_alteracoes = relatorio_alteracoes.merge(unidades, left_on='cod_empresa', right_on='Código', how='left')
relatorio_alteracoes = relatorio_alteracoes.merge(unidades, left_on='cod_empresa_anterior', right_on='Código', how='left', suffixes=('_destino', '_origem'))

relatorio_alteracoes = relatorio_alteracoes.rename(columns={
    'nome': 'Colaborador',
    'Empresa Resumo_origem': 'Unidade de origem',
    'cargo_anterior_anterior': 'Cargo de origem', 
    'Empresa Resumo_destino': 'Unidade de destino',
    'cargo': 'Cargo de destino'
})
relatorio_alteracoes = relatorio_alteracoes[['Registro', 'Colaborador', 'Unidade de origem', 'Cargo de origem', 'Unidade de destino', 'Cargo de destino']]

# 3.3 - Alterações Consolidadas
alteracoes_unidades_filtered = alteracoes_unidades[
    (pd.to_datetime(alteracoes_unidades['data_alteracao']).dt.year == ano) &
    (pd.to_datetime(alteracoes_unidades['data_alteracao']).dt.month == mes)
].copy()
alteracoes_unidades_filtered = alteracoes_unidades_filtered[['Registro', 'colaborador', 'unidade_origem', 'cargo_origem', 'unidade_destino', 'cargo_destino']]
alteracoes_unidades_filtered.columns = ['Registro', 'Colaborador', 'Unidade de origem', 'Cargo de origem', 'Unidade de destino', 'Cargo de destino']

alteracoes_final = pd.concat([relatorio_alteracoes, alteracoes_unidades_filtered], ignore_index=True)

# Controle de atualização de processo: Etapa 5
linha_5 = [id, datetime.today(), 5]
ws_p.append(linha_5)
wb_p.save(path_registros_processos)

# Controle de atualização de processo: Etapa 6
linha_6 = [id, datetime.today(), 6]
ws_p.append(linha_6)
wb_p.save(path_registros_processos)

c:\Users\rodrigo.bernandes\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


# Gerando os Relatórios

In [8]:
path_desligados = r'X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.1 - Relatório mensal Controladoria\10.1.5 - Base de Desligamentos\2026'
relatorio_desligados.to_excel(os.path.join(path_desligados, f'DESLIGADOS {mes_atual}-{ano}.xlsx'), index=False)

path_afastados = r'X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.1 - Relatório mensal Controladoria\10.1.2 - Base de Afastados\2026'
relatorio_afastados.to_excel(os.path.join(path_afastados, f'AFASTADOS {mes_atual}-{ano}.xlsx'), index=False)

path_alteracoes = r'X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.1 - Relatório mensal Controladoria\10.1.4 - Base de alterações funcionais\2026'
alteracoes_final.to_excel(os.path.join(path_alteracoes, f'ALTERAÇÕES {mes_atual}-{ano}.xlsx'), index=False)

# Controle de atualização de processo: Etapa 7
linha_7 = [id, datetime.today(), 7]
ws_p.append(linha_7)
wb_p.save(path_registros_processos)

# Resumo de Finalização do Processo

In [9]:
print('----------------------------------------------------------------------------------------------------')
print('')
print('   Processo finalizado')
print('')
print('   Tempo de execução:')
print('')
print(f'   {linha_7[1] - linha_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   Processo finalizado

   Tempo de execução:

   0:01:15.115380

----------------------------------------------------------------------------------------------------
